# cloudflare
> Cloudflare DNS and tunnel management via the official Python SDK

In [ ]:
#| default_exp cloudflare

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
from cloudflare import Cloudflare
from fastcore.all import L, first, patch
from fastops.core import secret_get

## CF

Thin wrapper around the official `cloudflare` Python SDK. Token is read from the `CLOUDFLARE_API_TOKEN` environment variable by default.

In [ ]:
#| export
class CF:
    'Cloudflare API wrapper using the official cloudflare Python SDK'
    def __init__(self, token=None, timeout=10, **kw):
        self._c = Cloudflare(api_token=token or secret_get('CLOUDFLARE_API_TOKEN'), timeout=timeout, **kw)

    def zones(self) -> list:
        'List all zones'
        return [z.model_dump() for z in self._c.zones.list()]

    def zone_id(self, domain) -> str:
        'Get zone_id for domain'
        for z in self._c.zones.list(name=domain): return z.id
        raise ValueError(f'Zone not found: {domain}')

    def dns_records(self, zone_id) -> list:
        'List DNS records for a zone'
        return [r.model_dump() for r in self._c.dns.records.list(zone_id=zone_id)]

    def create_record(self, zone_id, type, name, content, ttl=1, proxied=False) -> dict:
        'Create a DNS record'
        r = self._c.dns.records.create(zone_id=zone_id, type=type, name=name, content=content, ttl=ttl, proxied=proxied)
        return r.model_dump()

    def delete_record(self, zone_id, record_id) -> None:
        'Delete a DNS record'
        self._c.dns.records.delete(record_id, zone_id=zone_id)

    def upsert_record(self, domain, name, content, type='A', proxied=False) -> dict:
        'Create or replace DNS record. Deletes existing same-name+type record first.'
        zid = self.zone_id(domain)
        f = name if '.' in name else f'{name}.{domain}'
        # SDK supports filtering — huge perf win
        for r in self._c.dns.records.list(zone_id=zid, name=f, type=type):
	        self.delete_record(zid, r.id)
        return self.create_record(zid, type, name, content, proxied=proxied)

    def account_id(self) -> str:
        'Get first account ID'
        for a in self._c.accounts.list(): return a.id
        raise ValueError('No accounts found')

    def tunnels(self, account_id=None) -> list:
        'List Cloudflare tunnels'
        aid = account_id or self.account_id()
        return [t.model_dump() for t in self._c.zero_trust.tunnels.cloudflared.list(account_id=aid)]

    def create_tunnel(self, name, account_id=None) -> dict:
        'Create a new Cloudflare tunnel'
        aid = account_id or self.account_id()
        t = self._c.zero_trust.tunnels.cloudflared.create(account_id=aid, name=name)
        return t.model_dump()

    def delete_tunnel(self, tunnel_id, account_id=None) -> None:
        'Delete a Cloudflare tunnel'
        aid = account_id or self.account_id()
        self._c.zero_trust.tunnels.cloudflared.delete(tunnel_id, account_id=aid)

    def tunnel_token(self, tunnel_id, account_id=None) -> str:
        'Get tunnel token string for use with cloudflared_svc() / CF_TUNNEL_TOKEN env var'
        aid = account_id or self.account_id()
        result = self._c.zero_trust.tunnels.cloudflared.token.get(tunnel_id, account_id=aid)
        return result if isinstance(result, str) else result.token

    def tunnel_cname(self, domain: str, name: str, tunnel_id: str, proxied: bool = True) -> dict:
	    'Point subdomain at a tunnel (CNAME to *.cfargotunnel.com).'
	    return self.upsert_record(domain, name, f"{tunnel_id}.cfargotunnel.com",
	                              type='CNAME', proxied=proxied)

    def verify(self) -> dict:
        '''Check token validity and which permission groups are available (read-only, non-destructive).

        Required token permissions per capability:
        - dns_read:     Zone:DNS:Read  (or Zone:DNS:Edit)
        - account_read: Account:Account Settings:Read
        - tunnel_read:  Account:Cloudflare Tunnel:Edit  +  account_read
        '''
        try:
	        acc = self.account_id()
	        return dict(result=bool(self._c.zones.list() and self._c.zero_trust.tunnels.cloudflared.list(account_id=acc)))
        except Exception as e: return dict(result=False, err=str(e))

def dns_record(domain: str, name: str, content: str, type: str = 'A', proxied: bool = False, token: str | None = None) -> dict:
	'One-liner DNS (unchanged, just here for completeness).'
	return CF(token=token).upsert_record(domain, name, content, type=type, proxied=proxied)

def tunnel_record(domain: str, name: str, tunnel_id: str, proxied: bool = True, token: str | None = None) -> dict:
	'One-liner: point subdomain at an existing tunnel.'
	return CF(token=token).tunnel_cname(domain, name, tunnel_id, proxied=proxied)

@patch
def setup_tunnel(self:CF, domain: str, name: str, tunnel_name: str | None = None, proxied: bool = True):
    ''' One-liner for VPS hosting. Creates (or reuses) a tunnel, sets up the CNAME subdomain, and returns (tunnel_id, token) ready for cloudflared_svc() in your docker-compose.

    Usage in fastops:
        tid, token = setup_tunnel("example.com", "app")   # → app.example.com
        # then in Compose:
        # .svc("cloudflared", **cloudflared_svc(token_env=...)) with CF_TUNNEL_TOKEN=token
    '''
    tname = tunnel_name or name
    exists = L.filter(lambda x: x.get('name') == tname)
    tid = first(t)['id'] if (t:=exists(self.tunnels())) else self.create_tunnel(tname)['id']
    self.tunnel_cname(domain, name, tid, proxied=proxied)
    return tid, self.tunnel_token(tid)

### Least-privilege token setup

Create a **Custom Token** in the Cloudflare dashboard (My Profile → API Tokens → Create Token):

| Capability | Required permission |
|---|---|
| `dns_read` / DNS ops | Zone → DNS → **Edit** (scope to specific zone) |
| `account_read` / tunnel ops | Account → Account Settings → **Read** |
| `tunnel_read` / tunnel ops | Account → Cloudflare Tunnel → **Edit** |

Use `CF().verify()` to confirm your token has the permissions it needs before running ops.

In [ ]:
#| eval: false
c = CF()
c.verify()

{'result': True}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

IndentationError: unexpected indent (<unknown>, line 30)